# Problem 1 — TRAIN + VAL multiclass extraction
Builds the full official training pool (800 normal + 810 anomalous = 1,610 videos),
splits it into TRAIN/VAL at the video level (stratified by category), assigns
**multiclass** weak labels (video's category, applied to every clip — UCF-Crime has
no per-clip ground truth for training videos, this is the standard weak-supervision
approach), and extracts VideoMAE features resumably.

Independent of the test notebook — run this and Problem 2 in parallel Kaggle sessions.

In [ ]:
# ============================================================
# Configuration -- IDENTICAL in both notebooks on purpose.
# These values match what already produced your existing test/*.npz files
# (4s clips, 2s stride / 50% overlap, 16 frames @ 4fps, CLS-token pooling,
# MCG-NJU/videomae-base-finetuned-kinetics). Do not change independently
# between the two notebooks or clip_id alignment breaks.
# ============================================================
from pathlib import Path
import os, re, json, math, random, shutil, hashlib, warnings
import numpy as np
import pandas as pd
import cv2

from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED); np.random.seed(SEED)

NUM_FRAMES   = 16
SAMPLE_FPS   = 4.0
CLIP_DURATION = NUM_FRAMES / SAMPLE_FPS   # 4.0 seconds
CLIP_STRIDE   = CLIP_DURATION / 2         # 2.0 seconds, 50% overlap
ANNOTATION_FPS = 30.0
OVERLAP_THRESHOLD = 0.50

MODEL_NAME = "MCG-NJU/videomae-base-finetuned-kinetics"

CATEGORIES = ["Normal","Abuse","Arrest","Arson","Assault","Burglary","Explosion",
              "Fighting","RoadAccidents","Robbery","Shooting","Shoplifting",
              "Stealing","Vandalism"]
CAT2IDX = {c: i for i, c in enumerate(CATEGORIES)}
IDX2CAT = {i: c for c, i in CAT2IDX.items()}

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT  = Path("/kaggle/working/ucf_videomae")
SPLIT_ROOT    = WORK_ROOT / "splits"
FEATURE_ROOT  = WORK_ROOT / "features"
METADATA_ROOT = WORK_ROOT / "metadata"
for p in [SPLIT_ROOT, FEATURE_ROOT, METADATA_ROOT]:
    p.mkdir(parents=True, exist_ok=True)
for split in ["train", "val", "test"]:
    (FEATURE_ROOT / split).mkdir(parents=True, exist_ok=True)

VIDEO_EXTENSIONS = {".mp4", ".avi", ".mov", ".mkv"}
print("Config loaded. Categories:", CATEGORIES)


In [ ]:
# ============================================================
# Locate the UCF-Crime dataset root (same auto-discovery as the reference notebook)
# ============================================================
def find_ucf_root(base_dir=INPUT_ROOT):
    candidates = []
    for p in base_dir.iterdir():
        if not p.is_dir():
            continue
        has_anomaly_train = any(x.name.lower() == "anomaly_train.txt" for x in p.rglob("*") if x.is_file())
        has_temporal = any("temporal_anomaly_annotation" in x.name.lower() for x in p.rglob("*") if x.is_file())
        if has_anomaly_train and has_temporal:
            candidates.append(p)
    if not candidates:
        raise FileNotFoundError("Could not automatically find the UCF-Crime dataset under /kaggle/input.")
    return candidates[0]

DATASET_ROOT = find_ucf_root()
print("DATASET_ROOT:", DATASET_ROOT)
for item in sorted(DATASET_ROOT.iterdir()):
    print(("DIR " if item.is_dir() else "FILE"), item.name)


In [ ]:
# ============================================================
# Path-resolution helpers (identical to the reference notebook)
# ============================================================
def find_files_by_name(root, filename):
    return [p for p in root.rglob("*") if p.is_file() and p.name.lower() == filename.lower()]

def find_dirs_by_name(root, names):
    target = {n.lower() for n in names}
    return [p for p in root.rglob("*") if p.is_dir() and p.name.lower() in target]

all_video_paths = [p for p in DATASET_ROOT.rglob("*") if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS]
print(f"Total video files discovered: {len(all_video_paths):,}")

by_filename, by_relpath = {}, {}
for p in all_video_paths:
    by_filename.setdefault(p.name.lower(), []).append(p)
    rel = str(p.relative_to(DATASET_ROOT)).replace("\\", "/").lower()
    by_relpath[rel] = p

def normalize_split_line(line):
    line = line.strip().replace("\\", "/")
    if not line or line.startswith("#"):
        return None
    return line.strip()

def resolve_split_entry(entry):
    entry = normalize_split_line(entry)
    if entry is None:
        return None
    entry_low = entry.lower().lstrip("./")
    matches = [p for rel, p in by_relpath.items() if rel.endswith(entry_low)]
    if matches:
        return matches[0]
    filename = Path(entry).name.lower()
    candidates = by_filename.get(filename, [])
    if len(candidates) == 1:
        return candidates[0]
    if len(candidates) > 1:
        entry_parent = Path(entry).parent.name.lower()
        for p in candidates:
            if p.parent.name.lower() == entry_parent:
                return p
        return candidates[0]
    return None

def read_split_file(path):
    entries, unresolved = [], []
    with open(path, "r", errors="ignore") as fh:
        for line in fh:
            e = normalize_split_line(line)
            if e is None:
                continue
            resolved = resolve_split_entry(e)
            if resolved is not None:
                entries.append(resolved)
            else:
                unresolved.append(e)
    return entries, unresolved

def category_from_path(path):
    return path.parent.name

def video_id(video_path):
    # Hash the relative path so duplicate filenames in different folders never collide.
    # MUST stay identical across both notebooks -- this is what makes feature filenames
    # match up with whatever previously extracted the test set.
    rel = str(Path(video_path).relative_to(DATASET_ROOT))
    return hashlib.md5(rel.encode("utf-8")).hexdigest()[:16]


## Build the official TRAIN pool (anomaly + normal) with multiclass category labels

In [ ]:
anomaly_train_file = find_files_by_name(DATASET_ROOT, "Anomaly_Train.txt")
assert anomaly_train_file, "Anomaly_Train.txt not found"
ANOMALY_TRAIN_FILE = anomaly_train_file[0]
print("Anomaly_Train.txt:", ANOMALY_TRAIN_FILE)

anomaly_train_paths, anomaly_train_unresolved = read_split_file(ANOMALY_TRAIN_FILE)
print("Official anomaly TRAIN videos resolved:", len(anomaly_train_paths), "(expect ~810)")
if anomaly_train_unresolved:
    print("Unresolved entries:", len(anomaly_train_unresolved), anomaly_train_unresolved[:10])

normal_train_dirs = find_dirs_by_name(DATASET_ROOT, ["Training-Normal-Videos-Part-1", "Training-Normal-Videos-Part-2"])
normal_train_paths = sorted(set(p for d in normal_train_dirs for p in d.rglob("*")
                                 if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS))
print("Normal TRAIN videos:", len(normal_train_paths), "(expect ~800)")

# Safety check: make sure none of these accidentally overlap the official TEST list,
# if that file happens to be present in this same dataset (not required for this notebook).
test_txt = find_files_by_name(DATASET_ROOT, "Anomaly_Test.txt")
if test_txt:
    test_paths, _ = read_split_file(test_txt[0])
    test_names = {p.name for p in test_paths}
    overlap = {p.name for p in anomaly_train_paths} & test_names
    print(f"Cross-check against Anomaly_Test.txt -- overlap found: {len(overlap)} (should be 0)")
    if overlap:
        print("[WARNING] These filenames appear in both train and test lists:", list(overlap)[:10])


In [ ]:
anomaly_train_df = pd.DataFrame({
    "video_path": [str(p) for p in anomaly_train_paths],
    "video_filename": [p.name for p in anomaly_train_paths],
    "category": [category_from_path(p) for p in anomaly_train_paths],
})
normal_train_df = pd.DataFrame({
    "video_path": [str(p) for p in normal_train_paths],
    "video_filename": [p.name for p in normal_train_paths],
    "category": ["Normal"] * len(normal_train_paths),
})

official_train_df = pd.concat([anomaly_train_df, normal_train_df], ignore_index=True)
official_train_df = official_train_df.drop_duplicates(subset="video_filename").reset_index(drop=True)

bad_cat = [c for c in official_train_df["category"].unique() if c not in CAT2IDX]
if bad_cat:
    print(f"[WARNING] Unrecognized category folder names: {bad_cat} -- check CATEGORIES list / folder naming.")
official_train_df = official_train_df[official_train_df["category"].isin(CAT2IDX)].reset_index(drop=True)

official_train_df["video_label"] = official_train_df["category"]
official_train_df["video_label_idx"] = official_train_df["category"].map(CAT2IDX)

print("Official train pool:", len(official_train_df), "(expect ~1610)")
print(official_train_df["category"].value_counts())


## Split TRAIN pool into TRAIN / VAL (video-level, stratified by category)

In [ ]:
VAL_FRACTION = 0.15

cat_counts = official_train_df["category"].value_counts()
rare_cats = cat_counts[cat_counts < 2].index.tolist()
if rare_cats:
    print(f"[WARNING] Categories with <2 videos can't be stratified: {rare_cats} -- falling back to "
          f"binary (Normal vs Anomaly) stratification instead.")
    strat_col = (official_train_df["category"] == "Normal").astype(int)
else:
    strat_col = official_train_df["category"]

train_idx, val_idx = train_test_split(
    official_train_df.index, test_size=VAL_FRACTION, random_state=SEED, stratify=strat_col
)
train_df = official_train_df.loc[train_idx].copy().reset_index(drop=True)
val_df   = official_train_df.loc[val_idx].copy().reset_index(drop=True)
train_df["split"] = "train"
val_df["split"] = "val"

assert set(train_df["video_filename"]) & set(val_df["video_filename"]) == set(), "Leakage between train/val!"
print("TRAIN videos:", len(train_df), " VAL videos:", len(val_df))
print("\nTRAIN category counts:"); print(train_df["category"].value_counts())
print("\nVAL category counts:");   print(val_df["category"].value_counts())

train_df.to_csv(SPLIT_ROOT / "train_videos.csv", index=False)
val_df.to_csv(SPLIT_ROOT / "val_videos.csv", index=False)


## Build clip-level metadata (weak multiclass label = video's category)

In [ ]:
def get_video_info(video_path):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width, height = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    duration = frame_count / fps if fps and fps > 0 else None
    return {"fps": fps, "frame_count": frame_count, "width": width, "height": height, "duration_sec": duration}

def generate_clip_metadata_for_video(row):
    info = get_video_info(row["video_path"])
    duration = info["duration_sec"]
    if duration is None or duration < CLIP_DURATION:
        return []
    clips, start, clip_id = [], 0.0, 0
    while start + CLIP_DURATION <= duration + 1e-6:
        end = start + CLIP_DURATION
        clips.append({
            "video_path": row["video_path"], "video_filename": row["video_filename"],
            "category": row["category"], "video_label_idx": int(row["video_label_idx"]),
            "split": row["split"], "clip_id": clip_id,
            "start_sec": round(start, 4), "end_sec": round(end, 4),
            "video_fps": info["fps"], "video_duration_sec": duration,
            "width": info["width"], "height": info["height"],
            # Weak label: every clip inherits the video's category (no per-clip
            # ground truth exists for training videos -- known UCF-Crime limitation).
            "clip_label_idx": int(row["video_label_idx"]),
            "clip_label_name": row["category"],
            "annotation_overlap": np.nan,
            "label_source": "video_level_weak",
        })
        clip_id += 1
        start += CLIP_STRIDE
    return clips

def build_clip_metadata(video_df):
    records = []
    for i, row in video_df.iterrows():
        records.extend(generate_clip_metadata_for_video(row))
        if (i + 1) % 100 == 0:
            print(f"  processed {i+1}/{len(video_df)} videos")
    return pd.DataFrame(records)

train_clips = build_clip_metadata(train_df)
val_clips   = build_clip_metadata(val_df)
print("Train clips:", len(train_clips), " Val clips:", len(val_clips))
print(train_clips["clip_label_name"].value_counts())


In [ ]:
assert set(train_clips["video_filename"]) & set(val_clips["video_filename"]) == set(), "Leakage!"
train_clips.to_parquet(METADATA_ROOT / "train_clips_multiclass.parquet", index=False)
val_clips.to_parquet(METADATA_ROOT / "val_clips_multiclass.parquet", index=False)
print("Saved multiclass clip metadata for train/val.")


## Feature extraction (resumable -- run `restore_previous_features` first if you have an earlier output attached)

In [ ]:
# ============================================================
# Resume support: restore any previously-extracted features for this split
# if you attached an earlier output as a Kaggle input dataset.
# Safe to run even if nothing is found -- just prints 0 copied.
# ============================================================
def restore_previous_features(split, search_name="ucf_videomae/features"):
    candidates = list(Path("/kaggle/input").rglob(f"{search_name}/{split}"))
    dest = FEATURE_ROOT / split
    dest.mkdir(parents=True, exist_ok=True)
    copied = 0
    for cand in candidates:
        for src in cand.glob("*.npz"):
            dst = dest / src.name
            if not dst.exists():
                shutil.copy2(src, dst)
                copied += 1
    print(f"[{split}] restored {copied} previously-extracted feature files "
          f"(total now on disk: {len(list(dest.glob(\'*.npz\')))})")

restore_previous_features("train")
restore_previous_features("val")


In [ ]:
# ============================================================
# VideoMAE loading + feature extraction (identical scheme to what produced
# your existing features -- CLS-token pooling from the last hidden state)
# ============================================================
import torch
from transformers import VideoMAEImageProcessor, VideoMAEForVideoClassification

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

processor = VideoMAEImageProcessor.from_pretrained(MODEL_NAME)
model = VideoMAEForVideoClassification.from_pretrained(MODEL_NAME).to(DEVICE).eval()
print("Model loaded. hidden_size:", model.config.hidden_size)

def read_clip_frames(video_path, start_sec, end_sec, num_frames=NUM_FRAMES, sample_fps=SAMPLE_FPS):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open: {video_path}")
    video_fps = cap.get(cv2.CAP_PROP_FPS)
    if video_fps <= 0:
        cap.release()
        raise RuntimeError(f"Invalid FPS for: {video_path}")
    timestamps = start_sec + np.arange(num_frames) / sample_fps
    frames = []
    for t in timestamps:
        frame_idx = int(round(t * video_fps))
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ok, frame = cap.read()
        if not ok:
            cap.release()
            raise RuntimeError(f"Could not read frame {frame_idx} at {t:.3f}s from {video_path}")
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()
    if len(frames) != num_frames:
        raise RuntimeError(f"Expected {num_frames} frames, got {len(frames)}")
    return frames

def pool_videomae_output(outputs):
    hidden = outputs.hidden_states[-1]
    return hidden[:, 0, :] if hidden.ndim == 3 else hidden  # CLS token

def extract_one_clip_feature(video_path, start_sec, end_sec):
    frames = read_clip_frames(video_path, start_sec, end_sec)
    inputs = processor(frames, return_tensors="pt")
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.no_grad():
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(DEVICE.type == "cuda")):
            outputs = model(**inputs, output_hidden_states=True, return_dict=True)
    return pool_videomae_output(outputs).squeeze(0).float().cpu().numpy()

def extract_video_features(video_clip_df, split):
    assert len(video_clip_df) > 0
    video_path = video_clip_df.iloc[0]["video_path"]
    vid = video_id(video_path)
    out_file = FEATURE_ROOT / split / f"{vid}.npz"
    if out_file.exists():
        return "skipped", out_file

    feature_list, clip_ids, starts, ends = [], [], [], []
    for _, row in video_clip_df.iterrows():
        try:
            feat = extract_one_clip_feature(row["video_path"], float(row["start_sec"]), float(row["end_sec"]))
            feature_list.append(feat)
            clip_ids.append(int(row["clip_id"]))
            starts.append(float(row["start_sec"]))
            ends.append(float(row["end_sec"]))
        except Exception as e:
            print(f"ERROR: {row['video_filename']} clip {row['clip_id']}: {e}")
            raise

    features = np.stack(feature_list).astype(np.float32)
    np.savez(out_file, features=features, clip_id=np.array(clip_ids),
              start_sec=np.array(starts), end_sec=np.array(ends), video_id=vid)
    return "saved", out_file

def group_clips_by_video(clip_df):
    return [group for _, group in clip_df.groupby("video_filename", sort=False)]

def run_split_extraction(clip_df, split):
    groups = group_clips_by_video(clip_df)
    saved = skipped = 0
    for i, group in enumerate(groups, 1):
        status, _ = extract_video_features(group, split)
        saved += status == "saved"
        skipped += status == "skipped"
        if i % 10 == 0 or i == len(groups):
            print(f"{split}: {i}/{len(groups)} videos | saved={saved}, skipped={skipped}")
    return saved, skipped


In [ ]:
# Run when ready -- resumable, safe to interrupt and re-run.
train_saved, train_skipped = run_split_extraction(train_clips, "train")


In [ ]:
val_saved, val_skipped = run_split_extraction(val_clips, "val")


In [ ]:
# Final sanity summary
for split, clips in [("train", train_clips), ("val", val_clips)]:
    n_files = len(list((FEATURE_ROOT / split).glob("*.npz")))
    print(f"{split}: {clips['video_filename'].nunique()} videos in metadata, {n_files} feature files on disk")
print("\nRemember: labels live in the parquet files (clip_label_idx), not embedded per-file assumptions --")
print("join features to labels by (video_filename, clip_id) in the classification notebook.")
